In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.io import arff

# Import SMOTE library , the standard sklearn pipeline cannot be used here
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from datetime import datetime
from timeit import default_timer as timer

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder,MinMaxScaler
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV,StratifiedKFold, RandomizedSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.svm import SVC, OneClassSVM
from sklearn.ensemble import RandomForestClassifier, IsolationForest, GradientBoostingClassifier

from sklearn.metrics import (accuracy_score, recall_score,f1_score,RocCurveDisplay,roc_auc_score, \
PrecisionRecallDisplay, precision_score, precision_recall_curve, classification_report, ConfusionMatrixDisplay, confusion_matrix)


In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

import pickle
import os

# Define the directory where you saved the files
save_dir = '/content/drive/MyDrive/saved_model_components_backup'

# Load the train/test splits
with open(os.path.join(save_dir, 'X_train.pkl'), 'rb') as f:
    loaded_X_train = pickle.load(f)
with open(os.path.join(save_dir, 'X_test.pkl'), 'rb') as f:
    loaded_X_test = pickle.load(f)
with open(os.path.join(save_dir, 'y_train.pkl'), 'rb') as f:
    loaded_y_train = pickle.load(f)
with open(os.path.join(save_dir, 'y_test.pkl'), 'rb') as f:
    loaded_y_test = pickle.load(f)

# Load the dictionary of best models
with open(os.path.join(save_dir, 'best_models.pkl'), 'rb') as f:
    loaded_best_models = pickle.load(f)

# Load the results DataFrame
# with open(os.path.join(save_dir, 'df_results_cv_df.pkl'), 'rb') as f:
#     loaded_df_results_cv_df = pickle.load(f)

print("All components loaded successfully!")

All components loaded successfully!


In [3]:
X_train = loaded_X_train
X_test = loaded_X_test
y_train = loaded_y_train
y_test = loaded_y_test
#Print  the items
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(226980, 30)
(56746, 30)
(226980,)
(56746,)


### **Local Outlier Factor (LOF)**

**Core Principle:** Local Outlier Factor (LOF) is an unsupervised anomaly detection algorithm that computes a local anomaly score for each data point. It measures the density deviation of a data point with respect to its neighbors. The idea is that normal data points are deeply embedded in their neighborhood, while anomalies have a significantly lower density than their neighbors.

**Mechanism:**
1.  **Reachability Distance:** For each point, LOF calculates a 'reachability distance' to its k-nearest neighbors. This distance is a smoothed version of the Euclidean distance, ensuring stability in density calculations.
2.  **Local Reachability Density (LRD):** For each point, the LRD is the inverse of the average reachability distance from its k-nearest neighbors. A low LRD indicates that a point is far from its neighbors and thus might be an outlier.
3.  **Local Outlier Factor:** The LOF score of a point is the ratio of its LRD to the average LRD of its k-nearest neighbors. An LOF score significantly greater than 1 suggests an anomaly, as the point is less dense than its neighbors. Values close to 1 indicate inliers.

**Strengths:** Effective for datasets where anomalies do not deviate globally but are anomalous with respect to their local neighborhood. It doesn't assume any particular data distribution.

**Weaknesses:** Can be sensitive to the `n_neighbors` parameter. Computationally more expensive than Isolation Forest for very large datasets.

In [7]:
# Combine train and test sets to apply unsupervised anomaly detection
X = pd.concat([loaded_X_train, loaded_X_test])
y = pd.concat([loaded_y_train, loaded_y_test])

print(f"Combined X shape: {X.shape}")
print(f"Combined y shape: {y.shape}")

Combined X shape: (283726, 30)
Combined y shape: (283726,)


In [ ]:
from sklearn.neighbors import LocalOutlierFactor

def evaluate_lof(X, y_true, contamination_value):
    print(f"\n--- Evaluating Local Outlier Factor with contamination={contamination_value} ---")
    # Initialize Local Outlier Factor model
    # `n_neighbors` can be tuned, a common range is 20 to 100
    lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination_value, novelty=False)

    # Fit and predict. -1 for outliers, 1 for inliers
    y_pred_lof_raw = lof.fit_predict(X)

    # Convert LOF output (-1 for anomaly, 1 for normal) to our target convention (1 for fraud, 0 for normal)
    y_pred_lof = np.where(y_pred_lof_raw == -1, 1, 0)

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred_lof)
    recall = recall_score(y_true, y_pred_lof)
    f1 = f1_score(y_true, y_pred_lof)
    precision = precision_score(y_true, y_pred_lof)
    roc_auc = roc_auc_score(y_true, y_pred_lof) # Note: ROC AUC for binary predictions

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_lof))

    # Confusion Matrix Display
    cm = confusion_matrix(y_true, y_pred_lof)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix - LOF (Contamination={contamination_value})')
    plt.show()

    return {
        'model': 'Local Outlier Factor',
        'contamination': contamination_value,
        'accuracy': accuracy,
        'recall': recall,
        'f1_score': f1,
        'precision': precision,
        'roc_auc': roc_auc
    }

# Evaluate with contamination='auto'
results_lof_auto = evaluate_lof(X, y, 'auto')

print("\n--- Evaluating Local Outlier Factor with contamination=0.002 ---")

# Evaluate with contamination=0.002
results_lof_002 = evaluate_lof(X, y, 0.002)


--- Evaluating Local Outlier Factor with contamination=auto ---


###**One Class SVM**
**Core Principle:** Unlike standard SVM, which classifies data with labels, the One-Class SVM is an unsupervised model that trains primarily on normal data. It maps data into a high-dimensional feature space to learn a decision boundary that encloses the normal points (5:52-6:37).

**Boundary Mechanism:** The algorithm attempts to maximize the distance between the origin and the decision boundary. Points falling inside the boundary are considered normal, while those outside are flagged as anomalies.

**Handling Deviations:** The variable  (xi) represents the allowable deviation. If a point falls slightly outside the boundary (where the function ), it may still be tolerated as normal; however, if a point significantly exceeds this threshold, it is classified as an outlier.